# Transforming the data ( for a specific pathway )
## To work with the setstreams application

In [2]:
import pandas as pd

file_name = "Pathways_s1.out"
df = pd.read_csv(file_name, sep='\t')
df

,Realization,utility,week,infra.
0,0,0,626,29
1,1,0,626,29
2,2,0,730,29
3,3,0,782,29
4,4,0,834,29
...,...,...,...,...
5388,997,5,887,29
5389,998,5,782,29
5390,998,5,2087,28
5391,999,5,730,29


In [58]:
def preProcessPathway(file_name, path_number=1):
    def week_to_period(week):
        year = (week // 52) + 1
        # Determine the 5-year period
        return f"{(year // 5) * 5}'s - {((year // 5) * 5) + 5 - 1}'s"
    df = pd.read_csv(file_name, sep='\t')
    # excluding all the utilities that are not utility 3
    df = df[df['utility'] == 3]
    # excluding all infra.30 due to error
    df = df[df['infra.'] != 30]
    df['period'] = df['week'].apply(week_to_period)
    return df,transform_data(df.head(10), path_number)
    

In [4]:
def transform_data(df, path_number=1):
    # Initialize result dictionary
    formatted_data = {}

    # Group the data by period
    grouped = df.groupby('period')

    # Iterate through each group (period)
    for period, group in grouped:
        # Get the list of all realizations for the current period and format them with 'Path {path_number}'
        realizations = [f"Path {path_number} {{{realization}}}" for realization in group['Realization']]

        # Get the list of unique infra values for the current period
        unique_infra = group['infra.'].unique()

        # Initialize the output for this period
        period_output = f',{",".join(realizations)}&&&'
        
        # Iterate through each unique infra
        for infra in unique_infra:
            # Create a binary list indicating if each realization corresponds to the current infra
            # binary_list = [str(int(infra_val == infra)) for infra_val in group['infra.']]
            binary_list = ['1' if infra_val == infra else '0' for infra_val in group['infra.']]
            # Append the infra and the corresponding binary list to the output
            period_output += f'{infra},{",".join(binary_list)},&&&'

        # Store the formatted output for the current period
        formatted_data[period] = period_output

    return formatted_data

In [157]:
data_0,path_0 = preProcessPathway("Pathways_s0.out",0)
path_0

{"10's - 14's": ',Path 0 {0}&&&20,1,&&&',
 "40's - 44's": ',Path 0 {0}&&&9,1,&&&'}

In [59]:
data_0,path_0 = preProcessPathway("Pathways_s0.out",0)
data_1,path_1 = preProcessPathway("Pathways_s1.out")
data_100,path_100 = preProcessPathway("Pathways_s100.out",100)

In [60]:
data_list = [path_0,path_1]
combined_output = combine_formatted_data(data_list)


In [61]:
# print("path 0")
# for period, value in path_0.items():
#     print(f'{{"{period}": "{value}"}},')

# print("path 1")
# for period, value in path_1.items():
#     print(f'{{"{period}": "{value}"}},')

print("both")
for period, value in combined_output.items():
    print(f'{{"{period}": "{value}"}},')

both
{"10's - 14's": ",Path 0 {0},Path 0 {1},Path 1 {0},Path 1 {1},Path 1 {12},Path 1 {14}&&&20,1,1&&&29,0,0,1,1,1,1"},
{"15's - 19's": ",Path 0 {5},Path 1 {2},Path 1 {3},Path 1 {4},Path 1 {10},Path 1 {11}&&&20,1&&&29,0,1,1,1,1,1"},
{"20's - 24's": ",Path 0 {4},Path 0 {7}&&&20,1,1"},
{"25's - 29's": ",Path 0 {4}&&&9,1"},
{"30's - 34's": ",Path 0 {3},Path 0 {4}&&&20,1,0&&&12,0,1"},
{"35's - 39's": ",Path 0 {6}&&&20,1"},
{"40's - 44's": ",Path 0 {0},Path 1 {4}&&&9,1&&&12,0,1"},


In [ ]:
{"10's - 14's": ",Path 0 {0},Path 0 {1},Path 1 {0},Path 1 {1},Path 1 {12},Path 1 {14}&&&20,1,1,0,0,0,0,&&&29,0,0,1,1,1,1,&&&"},
{"15's - 19's": ",Path 0 {5},Path 1 {2},Path 1 {3},Path 1 {4},Path 1 {10},Path 1 {11}&&&20,1,0,0,0,0,0,&&&29,0,1,1,1,1,1,&&&"},
{"20's - 24's": ",Path 0 {4},Path 0 {7}&&&20,1,1,&&&"},
{"25's - 29's": ",Path 0 {4}&&&9,1,&&&"},
{"30's - 34's": ",Path 0 {3},Path 0 {4}&&&20,1,0,&&&12,0,1,&&&"},
{"35's - 39's": ",Path 0 {6}&&&20,1,&&&"},
{"40's - 44's": ",Path 0 {0},Path 1 {4}&&&9,1,0,&&&12,0,1,&&&"},

In [47]:
def combine_formatted_data(data_list):
    # Initialize an empty dictionary to store the combined result
    combined_data = {}

    # Iterate through each dataset in the input list
    for data in data_list:
        for period, value in data.items():
            # Split the data into sections before "&&&" separators
            value_sections = value.split("&&&")
            realizations = value_sections[0].strip(",")  # First part is realizations
            infra_sections = value_sections[1:]  # Remaining parts are infra sections

            # Initialize the period in combined_data if not present
            if period not in combined_data:
                combined_data[period] = {"realizations": [], "infra": {}}

            # Append realizations from the current path to the combined data
            if realizations:
                new_realizations = realizations.split(",")
                combined_data[period]["realizations"].extend(new_realizations)

            # Process each infra section
            for infra_section in infra_sections:
                if infra_section.strip():  # Skip empty infra sections
                    infra_data = infra_section.split(",")
                    infra_key = infra_data[0]  # The infra number (like 12, 7, etc.)

                    # Filter out empty strings and convert binary values to integers
                    binary_values = [int(val) for val in infra_data[1:] if val.strip().isdigit()]

                    # Initialize binary values if infra_key is not present
                    if infra_key not in combined_data[period]["infra"]:
                        # Create a zero-initialized binary list based on the number of realizations
                        combined_binary_values = [0] * len(combined_data[period]["realizations"])
                        combined_data[period]["infra"][infra_key] = combined_binary_values

                    # Determine the position to add the binary values based on the path
                    if "Path 0" in realizations:
                        # If it's Path 0's infra, extend with zeros for Path 1
                        for i in range(len(binary_values)):
                            if i < len(combined_data[period]["infra"][infra_key]):
                                combined_data[period]["infra"][infra_key][i] = max(
                                    combined_data[period]["infra"][infra_key][i],
                                    binary_values[i]
                                )
                            else:
                                # Append if the index is out of range
                                combined_data[period]["infra"][infra_key].append(binary_values[i])

                    else:
                        # If it's Path 1's infra, add zeros in front for Path 0
                        end_index = len(combined_data[period]["realizations"]) - len(binary_values)
                        for i in range(len(binary_values)):
                            if end_index + i < len(combined_data[period]["infra"][infra_key]):
                                combined_data[period]["infra"][infra_key][end_index + i] = max(
                                    combined_data[period]["infra"][infra_key][end_index + i],
                                    binary_values[i]
                                )
                            else:
                                # Append if the index is out of range
                                combined_data[period]["infra"][infra_key].append(binary_values[i])

    # Create the final combined formatted output
    final_combined_data = {}
    for period, data in combined_data.items():
        # Prepare realizations
        realization_part = f',{",".join(data["realizations"])}'

        # Prepare infra parts
        infra_parts = []
        for infra_key, binary_vals in data["infra"].items():
            binary_vals_str = ",".join(map(str, binary_vals))
            infra_parts.append(f'{infra_key},{binary_vals_str}')

        # Join infra parts, avoiding empty sections
        infra_part = "&&&".join(infra_parts)

        # Combine realization and infra sections
        final_combined_data[period] = f'{realization_part}&&&{infra_part}' if infra_part else realization_part

    return final_combined_data

In [21]:
data_list = [path_0,path_1]
combined_output = combine_formatted_data(data_list)


In [163]:
for period, value in combined_output.items():
    print(f'{{"{period}": "{value}"}},')

{"10's - 14's": ",Path 0 {0},Path 1 {0},Path 1 {1}&&&20,1&&&29,1,1,"},
{"40's - 44's": ",Path 0 {0}&&&9,1"},


In [136]:
def format_single_file(df):
    # Initialize the result dictionary
    formatted_data = {}
    
    # Group the data by period
    grouped = df.groupby('period')

    # Iterate through each group (period)
    for period, group in grouped:
        # Get the list of all realizations for the current period (authors in your example)
        realizations = list(group['Realization'])
        
        # Concatenate realizations (authors) as a comma-separated string
        realization_part = ",".join([f'Path {path_number} {{{r}}}' for r in realizations])

        # Define the sections (similar to InfoVis, VAST, SciVis/Vis)
        unique_infra = group['infra.'].unique()
        infra_part = ""
        
        # Iterate through each unique infra
        for infra in unique_infra:
            # Create a list of binary values corresponding to whether each realization corresponds to this infra
            binary_list = [str(int(infra_val == infra)) for infra_val in group['infra.']]
            
            # Add the infra name followed by the binary list
            infra_part += f'Infra_{infra},{",".join(binary_list)},&&&'
        
        # Combine the realization part with the infra part
        formatted_output = f',{realization_part},&&&{infra_part}'
        
        # Store the formatted output in the result dictionary
        formatted_data[period] = formatted_output

    return formatted_data